In [ ]:
# ---- IMAGE2SPHERE, L = 6, on the SYMSOL-CHILE BOTTOM VIEW of the 7-gonal pyramid, with the scaling that let the
# cosine-attention GATr learn it (reports/cosdir_bottom_view.md): a 14x14 ResNet map (--i2s_grid 14: last ResNet stage at
# stride 1, dilation 2) projected onto 400 hemisphere points (--i2s_proj_rec_level 3; upstream: 7x7 map, 104 points),
# constant learning rate after 5 warm-up epochs, 120 epochs. The harmonic band limit stays at L = 6, so the 7-fold turn
# is still not representable; the question is whether the 6-7 degree level of the 7x7 runs
# (arynabiaroza/symsol-chile-bottom7-i2s-l6-l4: 6.49; ...-constlr-e120-l4: 6.74) moves. Model and recipe as those
# kernels, on 2xT4 instead of 4xL4 (same global batch 64). The snapshot is the cosine-attention one (it carries
# models/i2s_real.py); the flags are patched in by the cell below. ETA 2-3 h (not measured on T4). Requested by a human.
RUN_NAME = 'bottom7_i2s_l6_grid14_proj3_constlr'
WANDB_API_KEY = None
REPO = 'chagrygoris/GA-Research'
BRANCH = 'claude/symsol-symsol2-differences-2dukos'   # unused offline
ARGS = ['--model', 'i2s_real', '--encoder', 'resnet50', '--dataset', 'symsol', '--symsol_set', '7',
        '--symsol_train_views', '4500', '--symsol_test_views', '1000', '--no-use_warp', '--no-use_synth', '--num_workers', '4',
        '--no-ema', '--no-fixed_val_noise', '--lmax', '6', '--i2s_eval_rec_level', '4', '--i2s_grid', '14',
        '--i2s_proj_rec_level', '3', '--batch_size', '64', '--optimizer', 'adamw', '--lr', '1e-4', '--lr_scaling', 'sqrt',
        '--lr_schedule', 'step', '--lr_step_gamma', '1.0', '--warmup_epochs', '5', '--n_epochs', '120', '--ddp', '--num_gpus', '2',
        '--sync_bn', '--ddp_find_unused', '--seed', '0']
WATCHDOG_MINUTES = (240, 420)   # 2xT4: 185 s per epoch measured (timing kernel), 120 epochs = 6.2 h + evaluation   # (RTX, other GPU): ~2x the estimate; a stop keeps the per-epoch resume checkpoint
MARKERS = {'pose3d/pose3d/models/cos_attention.py': 'def replace_attention', 'pose3d/pose3d/models/gatr_denoiser.py': 'POSE3D_SYMSOL_COSMV',
           'pose3d/pose3d/datasets/symsol.py': 'symsol_chile_bottom', 'pose3d/pose3d/engine/resume.py': 'RESUMABLE_EXIT'}


In [ ]:
import glob, os, shutil, socket, subprocess, tarfile

def online(host='github.com', timeout=5):
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False

ONLINE = online()
# Fail fast without a GPU: an account whose competition grant is not active gets Kaggle's CPU image even though the
# push asked for an RTX Pro 6000 (seen on 2026-10-31), and the run would otherwise die minutes later in a CUDA call.
import torch
assert torch.cuda.is_available(), 'no CUDA GPU in this kernel (CPU image): the accelerator was not granted'
print('GPU:', torch.cuda.get_device_name(0))
print('internet:', ONLINE)
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
os.chdir('/kaggle/working')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
if markers:        # a repo snapshot dataset, auto-extracted by Kaggle
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    shutil.copytree(markers[0][:markers[0].index('/pose3d/pose3d/config.py')], '/kaggle/working/GA-Research')
elif tgzs:
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
else:
    assert ONLINE, 'no internet and no repo snapshot attached'
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, f'https://github.com/{REPO}.git'],
                       capture_output=True, text=True)
    print(r.stdout + r.stderr); r.check_returncode()
print(subprocess.run(['git', '-C', 'GA-Research', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
for path, marker in MARKERS.items():
    assert marker in open(f'/kaggle/working/GA-Research/{path}').read(), f'stale code: {marker!r} is not in {path}'


In [ ]:
# Patch the snapshot: --dircloud_nside and POSE3D_SYMSOL_COS_SCALE (the same edit as the repository's commit; every
# replacement is asserted to apply exactly once, so a different snapshot fails here instead of training something else).
"""Adds --dircloud_nside (HEALPix density of the spherecloud tokens) to a pose3d tree; with cos=True also makes the cosine
attention's start temperature settable (POSE3D_SYMSOL_COS_SCALE). Every edit must apply exactly once."""
import os, sys

def edit(path, old, new, count=1):
    s = open(path).read(); assert s.count(old) == count, (path, old, s.count(old)); open(path, "w").write(s.replace(old, new))

def apply(root, cos=False):
    p = lambda f: os.path.join(root, "pose3d", f)
    edit(p("config.py"), "    dircloud_scalars: int = 64\n",
         "    dircloud_scalars: int = 64\n"
         "    # With cond_tokens=spherecloud: HEALPix nside of the hemisphere the tokens sit on. 4: 88 points, at most 16 per\n"
         "    # ring around the optical axis (azimuthal frequencies below 8); 8: 368 points, up to 32 per ring.\n"
         "    dircloud_nside: int = 4\n")
    edit(p("models/__init__.py"), "        dircloud_scalars=fl.dircloud_scalars,\n",
         "        dircloud_scalars=fl.dircloud_scalars,\n        dircloud_nside=fl.dircloud_nside,\n")
    edit(p("models/clifford_flow.py"), "                 dircloud_scalars: int = 64,\n",
         "                 dircloud_scalars: int = 64,\n                 dircloud_nside: int = 4,\n", count=2)
    edit(p("models/clifford_flow.py"), "SphereCloudHead(backbone_channels, scalars=dircloud_scalars)",
         "SphereCloudHead(backbone_channels, scalars=dircloud_scalars, nside=dircloud_nside)")
    edit(p("models/clifford_flow.py"), "                dircloud_scalars=dircloud_scalars, dircloud_up_token=dircloud_up_token,\n",
         "                dircloud_scalars=dircloud_scalars, dircloud_nside=dircloud_nside, dircloud_up_token=dircloud_up_token,\n")
    edit(p("evaluate.py"), '        dircloud_scalars=saved.get("dircloud_scalars", 64),\n',
         '        dircloud_scalars=saved.get("dircloud_scalars", 64),\n        dircloud_nside=saved.get("dircloud_nside", 4),\n')
    if cos:
        edit(p("models/gatr_denoiser.py"), 'scale=30.0, fused=True)',
             'scale=float(os.environ.get("POSE3D_SYMSOL_COS_SCALE", "30")), fused=True)')




def apply_grid(root):
    p = lambda f: os.path.join(root, "pose3d", f)
    edit(p("config.py"), "    dircloud_nside: int = 4\n",
         "    dircloud_nside: int = 4\n"
         "    # With cond_tokens=spherecloud and a ResNet: side of the feature map the tokens are lifted from. 7: the ResNet's\n"
         "    # own map at 224 px. 14: its last stage runs at stride 1 with dilation 2 (models/encoders.dilate_last_stage; the\n"
         "    # pretrained weights are kept), so the map is 14x14 and the lift's Gaussian is narrowed in proportion.\n"
         "    dircloud_grid: int = 7\n")
    edit(p("models/__init__.py"), "        dircloud_nside=fl.dircloud_nside,\n",
         "        dircloud_nside=fl.dircloud_nside,\n        dircloud_grid=fl.dircloud_grid,\n")
    edit(p("models/clifford_flow.py"), "                 dircloud_nside: int = 4,\n",
         "                 dircloud_nside: int = 4,\n                 dircloud_grid: int = 7,\n", count=2)
    edit(p("models/clifford_flow.py"),
         "        self.backbone = build_encoder(encoder_type, pretrained=pretrained_backbone,\n"
         "                                      depth_anything_model=depth_anything_model)\n",
         "        self.backbone = build_encoder(encoder_type, pretrained=pretrained_backbone,\n"
         "                                      depth_anything_model=depth_anything_model)\n"
         "        if cond_tokens == \"spherecloud\" and dircloud_grid != 7:\n"
         "            from pose3d.models.encoders import dilate_last_stage\n"
         "            if dircloud_grid != 14:\n"
         "                raise ValueError(\"dircloud_grid must be 7 or 14\")\n"
         "            dilate_last_stage(self.backbone)\n")
    edit(p("models/clifford_flow.py"), "SphereCloudHead(backbone_channels, scalars=dircloud_scalars, nside=dircloud_nside)",
         "SphereCloudHead(backbone_channels, scalars=dircloud_scalars, nside=dircloud_nside,\n"
         "                                                grid=dircloud_grid)")
    edit(p("models/clifford_flow.py"), "dircloud_nside=dircloud_nside, dircloud_up_token=dircloud_up_token,\n",
         "dircloud_nside=dircloud_nside, dircloud_grid=dircloud_grid,\n                dircloud_up_token=dircloud_up_token,\n")
    edit(p("evaluate.py"), '        dircloud_nside=saved.get("dircloud_nside", 4),\n',
         '        dircloud_nside=saved.get("dircloud_nside", 4),\n        dircloud_grid=saved.get("dircloud_grid", 7),\n')
    edit(p("models/dircloud_head.py"), "    def __init__(self, c_in: int = 2048, scalars: int = 64, nside: int = 4):\n",
         "    def __init__(self, c_in: int = 2048, scalars: int = 64, nside: int = 4, grid: int = 7):\n")
    edit(p("models/dircloud_head.py"), "        self.lift = SphereLift(c_in, c_s=scalars, nside=nside)\n",
         "        # the lift's Gaussian (0.2 for the 7x7 map, in units of half the map) follows the cell size\n"
         "        self.lift = SphereLift(c_in, c_s=scalars, nside=nside, grid=grid, sigma=0.2 * 6 / (grid - 1))\n")
    edit(p("models/so3fourier_head.py"),
         "        if grid % 2 == 0 or grid < 3:\n"
         "            raise ValueError(\"grid must be odd (>= 3) so the centre cell sits on the optical axis\")\n",
         "        if grid < 3:\n"
         "            raise ValueError(\"grid must be >= 3\")\n")
    edit(p("models/so3fourier_head.py"),
         "        off = (torch.arange(grid, dtype=torch.float64) - grid // 2) / (grid // 2)\n",
         "        # cell centres in [-1, 1]; an odd grid has a centre cell on the optical axis, an even one straddles it\n"
         "        off = (torch.arange(grid, dtype=torch.float64) - (grid - 1) / 2) / ((grid - 1) / 2)\n")
    edit(p("models/encoders.py"), "def build_encoder(encoder_type: str, pretrained: bool = False,\n",
         "def dilate_last_stage(encoder):\n"
         "    \"\"\"A torchvision ResNet behind image2sphere's wrapper: run its last stage at stride 1 with dilation 2, as\n"
         "    torchvision's replace_stride_with_dilation does, so a 224 px image gives a 14x14 map instead of 7x7. The\n"
         "    weights are untouched, so pretrained ones stay valid.\"\"\"\n"
         "    net = getattr(encoder, \"encoder\", encoder)             # ImageNetNormalized wraps the ResNet\n"
         "    stage = net.layers[-1]\n"
         "    first = stage[0]\n"
         "    if not hasattr(first, \"conv3\"):\n"
         "        raise ValueError(\"dilate_last_stage expects a bottleneck ResNet (resnet50 / 101 / 152)\")\n"
         "    first.conv2.stride = (1, 1)\n"
         "    first.downsample[0].stride = (1, 1)\n"
         "    for block in list(stage)[1:]:\n"
         "        block.conv2.dilation, block.conv2.padding = (2, 2), (2, 2)\n"
         "    shape = (net.output_shape[0], 14, 14)\n"
         "    net.output_shape = shape\n"
         "    if net is not encoder:\n"
         "        encoder.output_shape = shape\n"
         "    return encoder\n\n\n"
         "def build_encoder(encoder_type: str, pretrained: bool = False,\n")


apply('/kaggle/working/GA-Research/pose3d', cos=True)
apply_grid('/kaggle/working/GA-Research/pose3d')


def apply_i2s(root):
    p = lambda f: os.path.join(root, "pose3d", f)
    edit(p("config.py"), "    i2s_normalize: bool = True\n",
         "    i2s_normalize: bool = True\n"
         "    # i2s_real: side of the ResNet feature map that is projected onto the sphere. 14 runs the ResNet's last stage at\n"
         "    # stride 1 with dilation 2 (models/encoders.dilate_last_stage) and narrows the projection's Gaussian to match.\n"
         "    i2s_grid: int = 7\n"
         "    # i2s_real: HEALPix level of the hemisphere points the map is projected onto (upstream: 2 = 104 points; 3 = 400).\n"
         "    i2s_proj_rec_level: int = 2\n")
    edit(p("models/__init__.py"),
         "    return I2SReal(\n        encoder_type=cfg.model.encoder,\n",
         "    return I2SReal(\n        grid=cfg.i2s.i2s_grid,\n        proj_rec_level=cfg.i2s.i2s_proj_rec_level,\n"
         "        encoder_type=cfg.model.encoder,\n")
    edit(p("models/i2s_real.py"), "        freeze_backbone: bool = False,\n",
         "        freeze_backbone: bool = False,\n        grid: int = 7,\n        proj_rec_level: int = 2,\n")
    edit(p("models/i2s_real.py"), "        self.frozen_backbone = bool(freeze_backbone)\n",
         "        if grid != 7 or proj_rec_level != 2:\n"
         "            # A finer feature map and / or more sphere points for the projection; the harmonic band limit lmax is\n"
         "            # unchanged. Upstream's projector is rebuilt for the new map size, its Gaussian (0.2 for 7x7, in units\n"
         "            # of half the map) scaled with the cell size.\n"
         "            from image2sphere.predictor import SpatialS2Projector\n"
         "            from pose3d.models.encoders import dilate_last_stage\n"
         "            if grid == 14:\n"
         "                dilate_last_stage(self.net.encoder)\n"
         "            elif grid != 7:\n"
         "                raise ValueError(\"i2s_grid must be 7 or 14\")\n"
         "            self.net.projector = SpatialS2Projector(tuple(self.net.encoder.output_shape), sphere_fdim, lmax,\n"
         "                                                    sigma=0.2 * 6 / (grid - 1), rec_level=proj_rec_level)\n"
         "        self.frozen_backbone = bool(freeze_backbone)\n")


apply_i2s('/kaggle/working/GA-Research/pose3d')

def apply_sync(root):
    """This snapshot's sync_buffers broadcasts every buffer as it is; NCCL takes only contiguous CUDA tensors and
    Image2Sphere stores its Wigner matrices transposed. The fix is the one on claude/symsol-symsol2-differences-2dukos."""
    edit(os.path.join(root, "pose3d", "engine", "distributed.py"),
         "    for buf in model.buffers():\n        dist.broadcast(buf, src=0)\n",
         "    gpu = torch.device(\"cuda\", torch.cuda.current_device()) if dist.get_backend() == \"nccl\" else None\n"
         "    for buf in model.buffers():\n"
         "        on_wrong_device = gpu is not None and buf.device.type != \"cuda\"\n"
         "        if buf.is_contiguous() and not on_wrong_device:\n"
         "            dist.broadcast(buf, src=0)\n"
         "        else:\n"
         "            tmp = buf.to(gpu if on_wrong_device else buf.device).contiguous()\n"
         "            dist.broadcast(tmp, src=0)\n"
         "            buf.copy_(tmp)\n")

apply_sync('/kaggle/working/GA-Research/pose3d')
print('patched: --dircloud_nside, --dircloud_grid, --i2s_grid, --i2s_proj_rec_level, sync_buffers')


In [ ]:
# Dependencies Kaggle lacks. Online: PyPI / GitHub, with the image's torch pinned by a constraints file.
# Offline: the wheels of the ga-research-offline-deps and ga-research-gatr-deps datasets (installed without
# dependencies: the image's torch must stay).
import importlib, importlib.metadata as md, sys
# ga-research-offline-deps holds the wheels and ResNet weights; ga-research-offline-deps-py313 adds the healpy
# wheel for the Python 3.13 image (October 2026). Every ga-research-offline-deps* folder attached is read.
deps = sorted(glob.glob('/kaggle/input/**/ga-research-offline-deps*', recursive=True))
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
if deps:
    wheels = sorted(w for d in deps for w in glob.glob(os.path.join(d, '*.whl')))
    if gatr_deps:
        wheels += sorted(glob.glob(os.path.join(gatr_deps[0], '*.whl')))
    # Keep only the wheels this image's Python can run: the datasets may hold one healpy build per Python version
    # (the Kaggle image moved from 3.12 to 3.13 in October 2026), and pip rejects the whole command on one bad wheel.
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags())
    wheels = [w for w in wheels if parse_wheel_filename(os.path.basename(w))[3] & supported]
    print('wheels:', [os.path.basename(w) for w in wheels])
    assert any('healpy' in w for w in wheels), 'no healpy wheel for this Python: attach a ga-research-offline-deps* dataset that has one'
    subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)
    ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(ckpt_dir, exist_ok=True)
    for w in glob.glob('/kaggle/input/**/resnet*-*.pth', recursive=True):   # ImageNet weights (resnet50, resnet101)
        shutil.copy(w, os.path.join(ckpt_dir, os.path.basename(w)))
else:
    assert ONLINE, 'no internet and no ga-research-offline-deps attached'
    cons = []
    for p in ['torch', 'torchvision', 'numpy', 'scipy', 'pillow', 'tqdm', 'wandb', 'scikit-image', 'matplotlib',
              'pandas', 'sympy', 'astropy', 'packaging']:
        try:
            cons.append(f'{p}=={md.version(p)}')
        except md.PackageNotFoundError:
            pass
    open('/kaggle/working/constraints.txt', 'w').write('\n'.join(cons) + '\n')
    subprocess.run(['pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt', 'e3nn==0.5.9', 'healpy==1.19.0',
                    'git+https://github.com/chagrygoris/image2sphere.git',
                    'git+https://github.com/chagrygoris/clifford-group-equivariant-neural-networks.git'], check=True)
    # GATr, the vector field: --no-deps, its pins would replace torch.
    subprocess.run(['pip', 'install', '-q', '--no-deps', 'einops', 'opt_einsum',
                    'git+https://github.com/Qualcomm-AI-research/geometric-algebra-transformer.git'], check=True)
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford']:
    importlib.import_module(mod)
# GATr hard-imports xformers (only for attention masks the flow never passes); the repo installs a stub before
# importing it (models/gatr_denoiser._import_gatr), so check the import the way the training process does.
sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
from pose3d.models.gatr_denoiser import _import_gatr
_import_gatr()
print('python', sys.version.split()[0], '| deps import cleanly')


In [ ]:
# W&B: live when online (the wandb_api_key secret, or WANDB_API_KEY above); offline files + WANDB_SYNC lines otherwise.
key = WANDB_API_KEY
if key is None and ONLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('wandb_api_key')
    except Exception as e:   # no secret attached: fall back to offline logging
        print('wandb_api_key secret not available:', e)
if key and ONLINE:
    import wandb
    wandb.login(key=key)
    os.environ['WANDB_API_KEY'] = key
else:
    os.environ['WANDB_MODE'] = 'offline'
print('wandb mode:', os.environ.get('WANDB_MODE', 'online'))


In [ ]:
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
DATA = '/kaggle/working/data'
# Render SYMSOL-CHILE (seeded: the same images in every kernel). Skipped when an earlier cell run already did it.
if not os.path.exists(f'{DATA}/symsol_chile_bottom/train/rotations.npz'):
    subprocess.run([sys.executable, '-u', '-m', 'pose3d.datasets.symsol_chile', f'--root={DATA}', '--train', '4500',
                    '--test', '1000', '--device', 'cuda', '--view', 'bottom', '--shapes', 'pyr7'], check=True)
print('SYMSOL-CHILE root:', DATA, flush=True)
# CPU preflight: the native model's own numerical tests (as the original kernel ran them), before any GPU time.
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests/test_native_stf.py',
                    'tests/test_symsol_chile.py', 'tests/test_resume.py'],
                   env={**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '4'}, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:], flush=True)
assert r.returncode == 0, 'CPU preflight failed'
name = RUN_NAME
os.environ['SYMSOL_SHARED_CACHE'] = '/kaggle/temp/symsol-shared-pixels'   # one image copy for all ranks
os.environ['POSE3D_SYMSOL_COSMV'] = '1'          # optimist1938's cosine attention, as in his kernel
os.environ['POSE3D_SYMSOL_ATTENTION'] = 'cosdir'
os.environ['POSE3D_SYMSOL_COS_SCALE'] = '60'   # start inverse temperature of the cosine attention (30 in the 88-point runs)
# RESUME (engine/resume.py): the run writes /kaggle/working/<run_name>_resume.pth after every epoch, which Kaggle keeps
# as this kernel's output. To continue a stopped run, launch a new kernel with the stopped one attached as input
# (gpu_pool/launcher.py --kernel-source <owner>/<kernel>): its checkpoint is found here and passed as --resume.
resume = sorted(glob.glob(f'/kaggle/input/**/{name}_resume.pth', recursive=True))
resume_args = []
if resume:
    shutil.copy(resume[0], f'/kaggle/working/{name}_resume.pth')     # keep advancing the same file
    resume_args = [f'--resume=/kaggle/working/{name}_resume.pth']
    print('resuming from', resume[0], flush=True)
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={DATA}', '--platform=kaggle', f'--run_name={name}',
       '--resume_dir=/kaggle/working', *resume_args, *ARGS]
print(' '.join(cmd), flush=True)
# The output is also written to /kaggle/working/<run_name>.log: Kaggle keeps /kaggle/working as the kernel's output
# even when its own log stream comes back empty (seen on long RTX sessions), so the per-epoch metrics survive.
# WATCHDOG: the training process is killed after MAX_MINUTES (WATCHDOG_MINUTES, about twice the estimate) so
# that a stuck session cannot burn the 12-hour GPU budget (two sessions did exactly that).
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
MAX_MINUTES = WATCHDOG_MINUTES[0 if 'RTX' in gpu_name else 1]
print(f'GPU: {gpu_name!r}, watchdog {MAX_MINUTES} min', flush=True)
import threading, time
with open(f'/kaggle/working/{name}.log', 'w') as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ})
    def _watchdog():
        time.sleep(MAX_MINUTES * 60)
        if proc.poll() is None:
            # SIGTERM first: the run exits with 75 and keeps its last epoch's resume checkpoint; SIGKILL only if stuck.
            print(f'WATCHDOG: stopping the run after {MAX_MINUTES} minutes (resumable)', flush=True)
            proc.terminate()
            try:
                proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                proc.kill()
    threading.Thread(target=_watchdog, daemon=True).start()
    for line in proc.stdout:
        print(line, end='', flush=True)
        fh.write(line); fh.flush()
    rc = proc.wait()
# 75: stopped (watchdog / SIGTERM) with a resume checkpoint in /kaggle/working; the kernel ends cleanly so its output
# is kept, and a new kernel with this one attached continues it. Anything else non-zero is a crash and fails the kernel.
if rc == 75:
    print(f'STOPPED, RESUMABLE: /kaggle/working/{name}_resume.pth; relaunch with this kernel as --kernel-source', flush=True)
else:
    assert rc == 0, f'pose3d exited with code {rc}'
